In [2]:
import numpy as np
from ase import Atoms
# If RDKit is available for molecule generation:
from rdkit import Chem
from rdkit.Chem import AllChem
import nglview as nv
# 1. Generate a 3D geometry for n-hexadecane (C16H34) using RDKit
smiles = "CCCCCCCCCCCCCCCC"  # 16 carbons chain
mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol)  # optimize geometry with MMFF94 force field
# Extract atomic symbols and coordinates from RDKit
symbols_single = [atom.GetSymbol() for atom in mol.GetAtoms()]
conf = mol.GetConformer()
coords_single = np.array([list(conf.GetAtomPosition(i)) for i in range(conf.GetNumAtoms())], dtype=float)
# Center the single molecule at the origin (using mass-weighted center of mass)
masses_single = np.array([atom.GetMass() for atom in mol.GetAtoms()])
com = np.average(coords_single, axis=0, weights=masses_single)
coords_single -= com  # shift center of mass to (0,0,0)

In [3]:
# 2. Replicate the molecule to build a liquid configuration
n_x, n_y, n_z = 4, 4, 3   # 4x4x3 = 48 molecules total
box_length = 28.5  # angstrom, choose to get ~0.78 g/cm^3 for 48 C16H34 molecules
all_symbols = []
all_positions = []
all_tags = []       # to label which atoms belong to which molecule
mol_index = 0
# Distribute molecules on a grid with some spacing

for i in range(n_x):
    for j in range(n_y):
        for k in range(n_z):
            mol_index += 1
            # Determine a roughly even-spaced position in the box for this molecule
            pos_center = np.array([ (i+1)*box_length/(n_x+1),
                                     (j+1)*box_length/(n_y+1),
                                     (k+1)*box_length/(n_z+1) ])
            # Random rotation for the molecule
            # (generate a random rotation matrix via axis-angle)
            axis = np.random.randn(3); axis /= np.linalg.norm(axis)
            angle = np.random.rand() * 360  # in degrees
            # Rotation matrix (Rodrigues' formula)
            theta = np.deg2rad(angle)
            c, s = np.cos(theta), np.sin(theta)
            ux, uy, uz = axis
            R = np.array([
                [c+ux**2*(1-c),   ux*uy*(1-c)-uz*s, ux*uz*(1-c)+uy*s],
                [uy*ux*(1-c)+uz*s, c+uy**2*(1-c),   uy*uz*(1-c)-ux*s],
                [uz*ux*(1-c)-uy*s, uz*uy*(1-c)+ux*s, c+uz**2*(1-c)]
            ])
            # Apply rotation and shift
            coords_rot = coords_single.dot(R.T)
            coords_new = coords_rot + pos_center
            # Append this molecule's atoms to the global list
            for atom_symbol, atom_pos in zip(symbols_single, coords_new):
                all_symbols.append(atom_symbol)
                all_positions.append(atom_pos)
                all_tags.append(mol_index)
# Create ASE Atoms object for the full system
atoms = Atoms(symbols=all_symbols, positions=np.array(all_positions),
              cell=[box_length, box_length, box_length], pbc=True)
atoms.set_tags(all_tags)
print(atoms)  # this will show 48*50 = 2400 atoms (each C16H34 has 50 atoms)
nv.show_ase(atoms)

Atoms(symbols='C768H1632', pbc=True, cell=[28.5, 28.5, 28.5], tags=...)


NGLWidget()

In [4]:
import numpy as np

# Convert lists to arrays for convenience
com_history = np.array(com_history)  # shape: (n_times, n_mol, 3)
time_history = np.array(time_history)  # in seconds (SI units)

# Use the first frame as reference (t=0)
ref_com = com_history[0]  # shape (n_mol, 3) at t=0
# Compute MSD at the final time (or as a function of time)
disp = com_history - ref_com  # displacement of each molecule from t=0
# Apply PBC unwrapping to displacement (already handled incrementally above, so this should be minimal)
# Calculate squared displacement for each molecule at each time:
sq_disp = np.sum(disp**2, axis=2)  # shape: (n_times, n_mol)
msd_vs_time = np.mean(sq_disp, axis=1)  # average over molecules

# Take the final time interval for diffusion (or fit slope):
total_disp = msd_vs_time[-1]  # MSD at final time
total_t = time_history[-1]    # total simulation time in s
D_est = total_disp / (6 * total_t)
print(f"Estimated diffusion coefficient: {D_est:.3e} m^2/s")


NameError: name 'com_history' is not defined

In [5]:
#!/usr/bin/env python3
"""
All-atom simulation of liquid n-hexadecane diffusion using OPLS-AA in ASE+LAMMPS.
"""
import numpy as np
from ase import Atoms
from ase.io import write
import os
from ase.calculators.lammpsrun import LAMMPS
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
import ase.units as units

# RDKit for initial molecule generation
from rdkit import Chem
from rdkit.Chem import AllChem

# --- 1) Define OPLS-AA Force Field Parameters for n-Alkanes ---
# These parameters are from the original OPLS-AA papers by Jorgensen et al.
# We will assign integer types for LAMMPS:
# 1: opls_135 (Alkane CH3 Carbon)
# 2: opls_136 (Alkane CH2 Carbon)
# 3: opls_140 (Alkane Hydrogen)

opls_parameters = {
    # Atom types, mass, charge, and LJ parameters (sigma, epsilon)
    'atom_info': {
        1: {'mass': 12.011, 'charge': -0.18, 'sigma': 3.50, 'epsilon': 0.066}, # CH3-C
        2: {'mass': 12.011, 'charge': -0.12, 'sigma': 3.50, 'epsilon': 0.066}, # CH2-C
        3: {'mass': 1.008,  'charge':  0.06, 'sigma': 2.50, 'epsilon': 0.030}, # H
    },
    # Bond parameters (K_b, r_0)
    'bonds': {
        'C-C': {'k': 268.0, 'r0': 1.529}, # kcal/mol/A^2
        'C-H': {'k': 340.0, 'r0': 1.090}, # kcal/mol/A^2
    },
    # Angle parameters (K_theta, theta_0)
    'angles': {
        'C-C-C': {'k': 58.35, 'theta0': 112.7}, # kcal/mol/rad^2
        'C-C-H': {'k': 37.5,  'theta0': 110.7},
        'H-C-H': {'k': 33.0,  'theta0': 107.8},
    },
    # Dihedral parameters (V1, V2, V3, V4) - OPLS style
    'dihedrals': {
        'C-C-C-C': {'v': [1.411, -0.271, 3.145, 0.0]}, # kcal/mol
        'H-C-C-C': {'v': [0.0, 0.0, 0.355, 0.0]},
        'H-C-C-H': {'v': [0.0, 0.0, 0.318, 0.0]},
    }
}

# --- 2) Generate a single All-Atom n-Hexadecane Molecule ---
smiles = "CCCCCCCCCCCCCCCC"
mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol)

symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
coords = np.array([mol.GetConformer().GetAtomPosition(i) for i in range(mol.GetNumAtoms())])
masses_single = np.array([atom.GetMass() for atom in mol.GetAtoms()])
com = np.average(coords, axis=0, weights=masses_single)
coords -= com  # shift center of mass to (0,0,0)
n_atoms_per_mol = len(symbols)

# Assign OPLS atom types and charges
atom_types = np.zeros(n_atoms_per_mol, dtype=int)
charges = np.zeros(n_atoms_per_mol)
for atom in mol.GetAtoms():
    idx = atom.GetIdx()
    sym = atom.GetSymbol()
    if sym == 'C':
        # Terminal CH3 carbon if it has only one carbon neighbor
        c_neighbors = [n.GetSymbol() for n in atom.GetNeighbors()].count('C')
        if c_neighbors == 1:
            atom_types[idx] = 1 # opls_135
            charges[idx] = opls_parameters['atom_info'][1]['charge']
        else:
            atom_types[idx] = 2 # opls_136
            charges[idx] = opls_parameters['atom_info'][2]['charge']
    elif sym == 'H':
        atom_types[idx] = 3 # opls_140
        charges[idx] = opls_parameters['atom_info'][3]['charge']
# Build single molecule Atoms object
single_mol = Atoms(symbols=symbols, positions=coords)
single_mol.set_array('atom_types', atom_types)
single_mol.set_array('charge', charges)

# --- 3) Build Liquid Box and Topology ---
# Get topology templates from the RDKit molecule
single_bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in mol.GetBonds()]
single_angles = []
for atom in mol.GetAtoms():
    idx = atom.GetIdx()
    neighbors = [nbr.GetIdx() for nbr in atom.GetNeighbors()]
    for i in range(len(neighbors)):
        for j in range(i + 1, len(neighbors)):
            single_angles.append((neighbors[i], idx, neighbors[j]))
single_dihedrals = []
for bond in mol.GetBonds():
    begin, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    begin_neighbors = [n.GetIdx() for n in mol.GetAtomWithIdx(begin).GetNeighbors() if n.GetIdx() != end]
    end_neighbors = [n.GetIdx() for n in mol.GetAtomWithIdx(end).GetNeighbors() if n.GetIdx() != begin]
    for i in begin_neighbors:
        for j in end_neighbors:
            single_dihedrals.append((i, begin, end, j))

n_x, n_y, n_z = 3, 3, 2  # 18 molecules to keep atom count reasonable
n_mol = n_x * n_y * n_z
box_length = 30.0  # Å, adjust for density (~0.77 g/cm^3)

all_atoms = Atoms()
all_bonds_list, all_angles_list, all_dihedrals_list = [], [], []

for i in range(n_x):
    for j in range(n_y):
        for k in range(n_z):
            global_atom_offset = len(all_atoms)
            mol_copy = single_mol.copy()
            mol_copy.rotate(np.random.rand() * 360, 'x', rotate_cell=False)
            mol_copy.rotate(np.random.rand() * 360, 'y', rotate_cell=False)
            mol_copy.rotate(np.random.rand() * 360, 'z', rotate_cell=False)
            
            offset = [i * box_length/n_x, j * box_length/n_y, k * box_length/n_z]
            mol_copy.translate(offset)
            all_atoms.extend(mol_copy)

            # Add topology for the new molecule using global indices
            for b1, b2 in single_bonds:
                all_bonds_list.append((b1 + global_atom_offset, b2 + global_atom_offset))
            for a1, a2, a3 in single_angles:
                all_angles_list.append((a1 + global_atom_offset, a2 + global_atom_offset, a3 + global_atom_offset))
            for d1, d2, d3, d4 in single_dihedrals:
                all_dihedrals_list.append((d1 + global_atom_offset, d2 + global_atom_offset, d3 + global_atom_offset, d4 + global_atom_offset))

all_atoms.set_cell([box_length] * 3)
all_atoms.set_pbc(True)
# Set molecule tags for analysis
all_atoms.set_tags(np.repeat(np.arange(1, n_mol + 1), n_atoms_per_mol))

# Attach the generated topology to the Atoms object
all_atoms.info['bonds'] = all_bonds_list
all_atoms.info['angles'] = all_angles_list
all_atoms.info['dihedrals'] = all_dihedrals_list
nv.show_ase(all_atoms)

NGLWidget()